# Code for GG4257: Urban Analytics as a Toolkit for Sustainable Urban Development

## Reproducing the Code

## Methodology

**Research Question 1: How do different types of crime vary across geodemographic neighbourhood types in London?**

- **Data Preparation:**
  - Disaggregated total crime into specific types: violent crime, property-related crime, anti-social behaviour, and drug-related crime.
- **Analysis:**
  - Cross-tabulated crime types against geodemographic clusters.
  - Visualised differences across clusters using boxplots and heatmaps.

**Research Question 2: How do average house prices vary across geodemographic clusters, and what is the relationship between housing affordability, crime rates, and neighbourhood types?**

- **Data Preparation:**
  - Merged 2020 and 2021 house price paid data with LSOA-level crime and cluster data.
  - Calculated percentage change in house prices (2020–2021).
- **Analysis:**
  - Correlation analysis between total crime, specific crime types, and 2021 house prices.
  - Scatterplots and heatmaps to visualise relationships.
  - Identification and mapping of LSOAs with high crime and low house prices.
  - Optional: simple linear regression models predicting house prices based on crime rates and cluster membership.

**Research Question 3: What geodemographic clusters exist across London, and how do they relate to the spatial distribution of crime?**

- **Data Preparation:**
  - Selected socio-economic indicators from the 2021 Census (e.g., age structure, ethnicity, tenure, education, employment status).
  - Z-score standardised all indicators.
- **Clustering:**
  - Applied K-Means clustering to define geodemographic neighbourhood types.
- **Spatial Analysis:**
  - Mapped geodemographic clusters across London.
  - Mapped total crime counts per Output Area.
  - Overlaid spatial patterns of crime onto cluster maps.
  - Calculated crime rates per Output Area to standardise for population differences.

In [ ]:
# Install dependencies 
! pip install -r requirements.txt

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from sodapy import Socrata
from shapely import wkt
import os
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist, pdist
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import mapclassify
import leafmap

### Data Collection and Preparation
- 2021 Census Output Area data (e.g., demographic, socio-economic indicators).
- Metropolitan and City of London Police 2024 crime data (aggregated to Output Areas and LSOAs).
- House Price Paid Data (2020 and 2021) aggregated at the LSOA level.
- LSOA boundary shapefiles (geometry data).
- Standardisation of variable names and datatypes (e.g., ensuring consistent LSOA codes across datasets).
- Aggregation of total crime counts and disaggregation into specific crime types (e.g., violent crime, ASB, drug-related, property-related).
- Spatial joining of crime and house price data to LSOA geometries.

### Output Area (OA) Data 

In [ ]:
# Load the bounds of Output Areas in London 
London_OA = "/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/raw/LOAC_OA_Shapefiles/LOAC_OA.shp"  
gdf_OA = gpd.read_file(London_OA)  

In [ ]:
gdf_OA.explore()

### Lower layer Super Output Areas (LSOA) Data 

In [ ]:
# Load the bounds of LSOA in London 
London_LSOA = "/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/raw/Lower_layer_Super_Output_Areas_(December_2021)/Lower_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFC_(V10).shp"  
gdf_LSOA = gpd.read_file(London_LSOA)  

In [ ]:
# Rename LSOA code column in the geometry GeoDataFrame
gdf_LSOA = gdf_LSOA.rename(columns={"LSOA21CD": "LSOA code"})

### Crime Data 

``` python

# Read and merge raw crime CSV files
csv_directory = "/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/raw/crime_data/"
csv_files = [file for file in os.listdir(csv_directory) if file.endswith(".csv")]
dataframes = []
for csv_file in csv_files:
    csv_path = os.path.join(csv_directory, csv_file)
    df = pd.read_csv(csv_path, low_memory=False)
    dataframes.append(df)
merged_data = pd.concat(dataframes, ignore_index=True)

# Save merged dataset to processed folder 
processed_directory = "/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/processed/"
output_path = os.path.join(processed_directory, "merged_crime_data.csv")
merged_data.to_csv(output_path, index=False)

```

In [ ]:
# Load processed crime data
crimes_df = pd.read_csv("/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/processed/merged_crime_data.csv")

In [ ]:
# Convert 'Month' column to datetime format for time-based analysis
crimes_df['Month'] = pd.to_datetime(crimes_df['Month'], format='%Y-%m')

In [ ]:
# Convert to GeoDataFrame with WGS84 coordinates
geometry = gpd.points_from_xy(crimes_df['Longitude'], crimes_df['Latitude'])
crimes_gdf = gpd.GeoDataFrame(crimes_df, geometry=geometry, crs="EPSG:4326")

In [ ]:
# Reproject crime data to match the Output Area CRS (EPSG:27700) for spatial join compatibility
crimes_gdf = crimes_gdf.to_crs(gdf_OA.crs)

In [ ]:
# Spatial join: match each crime to the OA polygon it falls within
crimes_with_OA = gpd.sjoin(crimes_gdf, gdf_OA, how="left", predicate="within")

In [ ]:
# Select relevant columns for analysis
relevant_columns = ['Crime ID', 'Month', 'Crime type', 'OA21CD', 'LSOA code', 'geometry']
crimes_with_OA = crimes_with_OA[relevant_columns]

# Remove any missing values 
crimes_with_OA = crimes_with_OA.dropna()

In [ ]:
# Remove "Other crime" rows
crimes_with_OA = crimes_with_OA[crimes_with_OA['Crime type'] != 'Other crime']

In [ ]:
# Map detailed crime types to broader crime categories and add as a new column
crime_group_map = {
    'Violence and sexual offences': 'Violence against the person',
    'Robbery': 'Violence against the person',
    
    'Burglary': 'Property-Related Crime',
    'Vehicle crime': 'Property-Related Crime',
    'Bicycle theft': 'Property-Related Crime',
    'Shoplifting': 'Property-Related Crime',
    'Criminal damage and arson': 'Property-Related Crime',
    'Theft from the person': 'Property-Related Crime',
    'Other theft': 'Property-Related Crime',

    'Public order': 'Anti-social Behaviours',
    'Possession of weapons': 'Anti-social Behaviours',

    'Drugs': 'Drug-related Crime'
}
crimes_with_OA['Crime_Group'] = crimes_with_OA['Crime type'].map(crime_group_map)

#### Crime Counts per OA

In [ ]:
# Aggregate crime counts by Output Area (OA) and crime group
crime_group_counts_OA = crimes_with_OA.groupby(['OA21CD', 'Crime_Group']).size().reset_index(name='Crime_Group_Count')

In [ ]:
# Pivot to wide format with separate columns for each crime group
crime_groups_OA = crime_group_counts_OA.pivot(index='OA21CD', columns='Crime_Group', values='Crime_Group_Count').fillna(0)

In [ ]:
# Calculate total crime per OA
crime_groups_OA['Total_Crime'] = (
    crime_groups_OA['Violence against the person'] +
    crime_groups_OA['Property-Related Crime'] +
    crime_groups_OA['Anti-social Behaviours'] +
    crime_groups_OA['Drug-related Crime']
)

In [ ]:
crime_groups_OA.head()

#### Crime Counts per LSOA

In [ ]:
# Aggregate crime counts by Output Area (OA) and crime group
crime_group_counts_LSOA = crimes_with_OA.groupby(['LSOA code', 'Crime_Group']).size().reset_index(name='Crime_Group_Count')

In [ ]:
# Pivot to wide format with separate columns for each crime group
crime_groups_LSOA = crime_group_counts_LSOA.pivot(index='LSOA code', columns='Crime_Group', values='Crime_Group_Count').fillna(0)

In [ ]:
# Calculate total crime per OA
crime_groups_LSOA['Total_Crime'] = (
    crime_groups_LSOA['Violence against the person'] +
    crime_groups_LSOA['Property-Related Crime'] +
    crime_groups_LSOA['Anti-social Behaviours'] +
    crime_groups_LSOA['Drug-related Crime']
)

In [ ]:
crime_groups_LSOA.head()

### House Prices Data

In [ ]:
# Load and clean processed house price data
house_prices_df = pd.read_csv("/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/raw/hpssa202103.csv")

In [ ]:
# Rename 'lsoacode' to 'LSOA code' in the house prices dataframe
house_prices_df = house_prices_df.rename(columns={"lsoacode": "LSOA code"})

### House Prices and Crime at LSOA level 

In [ ]:
# Now you can merge easily without specifying left_on and right_on
crime_house_prices = crime_groups_LSOA.merge(house_prices_df, on="LSOA code", how="left")

In [ ]:
# Merge the main dataframe with the geometry dataframe
crime_house_final = crime_house_prices.merge(gdf_LSOA[["LSOA code", "geometry"]], on="LSOA code", how="left")

# Turn it into a GeoDataFrame (so you can plot it easily)
crime_house_final = gpd.GeoDataFrame(crime_house_final, geometry="geometry")

crime_house_final.head()

### SES Data 

``` python 

# Set directory containing raw census CSVs
csv_directory = "/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/raw/census_raw_data/"

# List all CSV files in the directory
csv_files = [file for file in os.listdir(csv_directory) if file.endswith(".csv")]

# Initialize empty DataFrame for merged data
merged_data = pd.DataFrame()

# Read and merge all CSVs column-wise
for csv_file in csv_files:
    csv_path = os.path.join(csv_directory, csv_file)
    df_csv = pd.read_csv(csv_path, low_memory=False)
    merged_data = pd.concat([merged_data, df_csv], axis=1)  # Watch for duplicate OA codes

# Save merged census data to processed folder
merged_data.to_csv("/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/processed/merged_census_data.csv", index=False)

```

In [ ]:
# Load processed census data and merge with OA shapefile using OA codes
census_df = pd.read_csv("/Users/elenajarrett/Library/CloudStorage/OneDrive-Personal/Year 4/GG4257/Data IRP/Data/processed/merged_census_data.csv", low_memory=False)
merged_data_census = gdf_OA.merge(census_df, left_on='OA21CD', right_on='geography code', how='left')

### SES and Crime at OA level 

In [ ]:
# Merge crime and census data using OA code
crime_SES_final = merged_data_census.merge(crime_groups_OA, on='OA21CD', how='left').fillna(0)
crime_SES_final.head()

### Results

#### Research Question 1: Crime in London



In [ ]:
# Create a choropleth map using the 'Crime type' column
crimes_with_OA.sample(10000).explore(column='Crime type', cmap='RdYlBu', legend=True) 

In [ ]:
# Count incidents for all crime types
incident_counts = crimes_with_OA['Crime type'].value_counts()

# Plot all crime types
plt.figure(figsize=(12, 8))
sns.barplot(y=incident_counts.index, x=incident_counts.values, palette='Dark2')
plt.xlabel("Number of Crimes")
plt.ylabel("Crime Type")
plt.title("Crime Type Distribution in London")
plt.tight_layout()
plt.show()

In [ ]:
# Create a count per OA and Crime Type
crime_type_counts = crimes_with_OA.groupby(['OA21CD', 'Crime type']).size().reset_index(name='Count')

plt.figure(figsize=(16, 8))
sns.boxplot(
    data=crime_type_counts,
    x='Crime type',
    y='Count',
    palette="Set2",
    linewidth=1.2
)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.title('Distribution of Crime Counts per Output Area by Crime Type', fontsize=14, weight='bold')
plt.xlabel('Crime Type', fontsize=12)
plt.ylabel('Number of Crimes', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Create a count of each crime type
crime_counts = crimes_with_OA['Crime_Group'].value_counts().reset_index()
crime_counts.columns = ['Crime_Group', 'Count']

# Plot
plt.figure(figsize=(14, 8))
sns.barplot(
    data=crime_counts,
    x='Crime_Group',
    y='Count',
    hue='Crime_Group',     
    palette="Set3",
    legend=False            
)

plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.title('Number of Crimes by Group', fontsize=14, weight='bold')
plt.xlabel('Crime Group', fontsize=12)
plt.ylabel('Number of Crimes', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Classify 'crime_count' using different schemes
crime_series = crime_house_final["Total_Crime"]

# Natural Breaks (Jenks)
classifier_nb = mapclassify.NaturalBreaks(crime_series, k=5)

# Equal Interval
classifier_ei = mapclassify.EqualInterval(crime_series, k=5)

# Quantiles
classifier_qu = mapclassify.Quantiles(crime_series, k=5)

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram with Natural Breaks
sns.histplot(data=crime_house_final, x="Total_Crime", ax=axes[0], kde=True, bins=20)
for bin_value in classifier_nb.bins:
    axes[0].axvline(bin_value, color='red', linestyle='dashed', linewidth=2)
axes[0].set_title("Natural Breaks")
axes[0].legend(["Natural Breaks"])

# 2. Histogram with Equal Interval
sns.histplot(data=crime_house_final, x="Total_Crime", ax=axes[1], kde=True, bins=20)
for bin_value in classifier_ei.bins:
    axes[1].axvline(bin_value, color='blue', linestyle='dashed', linewidth=2)
axes[1].set_title("Equal Interval")
axes[1].legend(["Equal Interval"])

# 3. Histogram with Quantiles
sns.histplot(data=crime_house_final, x="Total_Crime", ax=axes[2], kde=True, bins=20)
for bin_value in classifier_qu.bins:
    axes[2].axvline(bin_value, color='green', linestyle='dashed', linewidth=2)
axes[2].set_title("Quantiles")
axes[2].legend(["Quantiles"])

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(18, 8))

# Natural Breaks Choropleth Map
crime_house_final.plot(
    column='Total_Crime',
    ax=axs[0],
    legend=True,
    cmap='viridis',
    scheme='UserDefined',
    classification_kwds={'bins': classifier_nb.bins}
)
axs[0].set_title("Choropleth Map of Crime Count with Natural Breaks")

# Equal Interval Choropleth Map
crime_house_final.plot(
    column='Total_Crime',
    ax=axs[1],
    legend=True,
    cmap='viridis',
    scheme='UserDefined',
    classification_kwds={'bins': classifier_ei.bins}
)
axs[1].set_title("Choropleth Map of Crime Count with Equal Intervals")

# Quantile Choropleth Map
crime_house_final.plot(
    column='Total_Crime',
    ax=axs[2],
    legend=True,
    cmap='viridis',
    scheme='UserDefined',
    classification_kwds={'bins': classifier_qu.bins}
)
axs[2].set_title("Choropleth Map of Crime Count with Quantiles")

plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()


In [ ]:
# Set the map center around Central London
london_crime_map = leafmap.Map(center=(51.5074, -0.1278), zoom=10, draw_control=False)

# Add a nice base map
london_crime_map.add_basemap("CartoDB.Positron")

# Define a style to remove the blue explore lines (no border)
style = {
    "color": "transparent",   # border color
    "weight": 0,              # border width
    "fillOpacity": 0.8        # fill transparency
}

# Optional: define hover style (if you want some feedback on hover)
hover_style = {
    "color": "#000000",       # on hover border color (set to transparent to disable)
    "weight": 1,
    "fillOpacity": 1.0
}

# Add Output Areas with crime counts (choropleth map)
london_crime_map.add_data(
    crime_house_final,
    column='Total_Crime',
    legend_title='Crime Count per LSOA',
    cmap='Reds',
    style=style,
    hover_style=hover_style  # Optional; remove if you don't want hover effects
)

# Display the interactive map
london_crime_map


In [ ]:
# List of crime type columns you want to plot
crime_types = ["Anti-social Behaviours", "Drug-related Crime", "Property-Related Crime", "Violence against the person"]

# Set up a 2x2 grid of plots (4 plots)
fig, axs = plt.subplots(2, 2, figsize=(18, 12))

# Flatten axs array for easy iteration
axs = axs.flatten()

# Loop through crime types and plot each one
for i, crime in enumerate(crime_types):
    crime_house_final.plot(
        column=crime,
        ax=axs[i],
        cmap='Reds',
        legend=True,
        scheme='Quantiles',  # <- use Quantiles classification
        k=5  # Number of quantile classes (you can adjust to k=4 or k=5)
    )
    axs[i].set_title(f"Choropleth of {crime} (Quantiles)")

# Clean layout
plt.tight_layout()
plt.show()


In [ ]:
# List of crime type columns you want to plot
crime_types = ["Anti-social Behaviours", "Drug-related Crime", "Property-Related Crime", "Violence against the person"]

# Set up a 2x2 grid of plots
fig, axs = plt.subplots(2, 2, figsize=(18, 12))
axs = axs.flatten()

# Loop through crime types and plot each one
for i, crime in enumerate(crime_types):
    # Create the Natural Breaks classifier manually for each crime type
    classifier_nb = mapclassify.NaturalBreaks(crime_house_final[crime], k=5)
    
    crime_house_final.plot(
        column=crime,
        ax=axs[i],
        cmap='viridis',
        legend=True,
        scheme='UserDefined',
        classification_kwds={'bins': classifier_nb.bins}
    )
    axs[i].set_title(f"Choropleth of {crime} (Natural Breaks)")

# Clean layout
plt.tight_layout()
plt.show()


#### Research Question 2: House Prices, Crime, and Cluster Membership
- Scatterplots: total crime vs house prices.
- Map of high-crime, low-house-price areas.

In [ ]:
# Create a filtered version of the GeoDataFrame where house prices are not missing
crime_house_final_price = crime_house_final.dropna(subset=["hpmd202003"])

# Set up the map
london_price_map = leafmap.Map(center=(51.5074, -0.1278), zoom=10, draw_control=False)
london_price_map.add_basemap("CartoDB.Positron")

# Define styles
style = {
    "color": "transparent",
    "weight": 0,
    "fillOpacity": 0.8
}
hover_style = {
    "color": "#000000",
    "weight": 1,
    "fillOpacity": 1.0
}

# Add data (no missing values now!)
london_price_map.add_data(
    crime_house_final_price,      # <-- use the filtered GeoDataFrame
    column='hpmd202003',
    legend_title='House Price (March 2020)',
    cmap='viridis',
    style=style,
    hover_style=hover_style
)

# Display map
london_price_map

In [ ]:
# Define high crime and low house price areas
high_crime_low_price = crime_house_final[
    (crime_house_final["Total_Crime"] > crime_house_final["Total_Crime"].quantile(0.75)) &
    (crime_house_final["hpmd202003"] < crime_house_final["hpmd202003"].median())
]

# Define low crime and high house price areas
low_crime_high_price = crime_house_final[
    (crime_house_final["Total_Crime"] < crime_house_final["Total_Crime"].quantile(0.25)) &
    (crime_house_final["hpmd202003"] > crime_house_final["hpmd202003"].median())
]

# Plot all LSOAs lightly in the background
base = crime_house_final.plot(color='lightgrey', edgecolor='white', figsize=(10, 10))

# Plot high crime, low house price areas in red
high_crime_low_price.plot(ax=base, color='red', edgecolor='black', label='High Crime, Low House Price')

# Plot low crime, high house price areas in blue
low_crime_high_price.plot(ax=base, color='blue', edgecolor='black', label='Low Crime, High House Price')

# Add a title
plt.title("High Crime & Low House Price vs Low Crime & High House Price LSOAs")

# Hide axis
plt.axis('off')

# Add a legend manually (since geopandas plot does not auto-add)
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color='red', label='High Crime, Low House Price')
blue_patch = mpatches.Patch(color='blue', label='Low Crime, High House Price')
plt.legend(handles=[red_patch, blue_patch], loc='lower left')

# Show the plot
plt.show()


In [ ]:
# Defining the total crimes and the house prices to use for the scatterplot
total_crimes = crime_house_final['Total_Crime']
house_prices_2021 = crime_house_final['hpmd202003']

# Creating a scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(x=total_crimes, y=house_prices_2021, scatter_kws={'alpha':0.3})
plt.title('Relationship Between Total Crime and House Prices (2020)')
plt.xlabel('Total Crime Count')
plt.ylabel('House Price (2020)')
plt.grid(True)
plt.show()

# Calculating and displaying the correlation coefficient
correlation = total_crimes.corr(house_prices_2021)
print("Correlation Coefficient:", correlation)

In [ ]:
crime_house_final[["Total_Crime", "hpmd202003"]].corr()

In [ ]:
# Select the crime columns and house price column
crime_type_cols = ["Anti-social Behaviours", "Drug-related Crime", "Property-Related Crime", "Violence against the person"]
house_price_column = "hpmd202003"

# Create a new DataFrame
crime_price_df = crime_house_final[crime_type_cols + [house_price_column]]

# Calculate the correlation matrix
corr = crime_price_df.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Between Crime Types and House Prices (2020)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### Research Question 3: Identification and Mapping of Geodemographic Clusters

#### Geodemographic Classification (SES)

In [ ]:
# Function to calculate percentage values for selected demographic and socio-economic indicators
# Each value column is divided by its corresponding total column to compute a new percentage feature
def calculate_percentages(dataframe, total_columns, value_columns):

    result_df = pd.DataFrame()

    for total_col, value_col in zip(total_columns, value_columns):
        percentage_col_name = f"{value_col}_percentage"

        if total_col not in dataframe.columns or value_col not in dataframe.columns:
            print(f"Warning: '{total_col}' or '{value_col}' not found in DataFrame. Skipping...")
            continue  # Skips missing columns instead of raising an error

        # Convert columns to numeric or NaN if errors occur
        dataframe[value_col] = pd.to_numeric(dataframe[value_col], errors='coerce')
        dataframe[total_col] = pd.to_numeric(dataframe[total_col], errors='coerce')
        
        result_df[percentage_col_name] = (dataframe[value_col] / dataframe[total_col]) * 100

    return result_df

# List of the corresponding totals 
total_cols = [
    # Housing / Accommodation
    'Accommodation type: Total: All households',
    'Accommodation type: Total: All households',
    'Accommodation type: Total: All households',
    'Accommodation type: Total: All households',

    # Deprivation
    'Household deprivation: Total: All households; measures: Value',
    
    # Commuting
    'Distance travelled to work: Total: All usual residents aged 16 years and over in employment the week before the census',
    'Distance travelled to work: Total: All usual residents aged 16 years and over in employment the week before the census',
    'Distance travelled to work: Total: All usual residents aged 16 years and over in employment the week before the census',
 
    # Tenure
    'Tenure of household: Total: All households',
    'Tenure of household: Total: All households',

    # Health
    'General health: Total: All usual residents',

    # Socioeconomic Classification 
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over',
    
    # Language / Migration
    'Proficiency in English language: Total: All usual residents aged 3 years and over',

    # Age
    'Age: Total',
    'Age: Total',
    'Age: Total',
    'Age: Total',

    # Economic Activity
    'Economic activity status: Total: All usual residents aged 16 years and over',
    'Economic activity status: Total: All usual residents aged 16 years and over',

    # Ethnicity
    'Ethnic group: Total: All usual residents',
    'Ethnic group: Total: All usual residents',
    'Ethnic group: Total: All usual residents',

    # Transport
    'Number of cars or vans: Total: All households',

    # Education
    'Highest level of qualification: Total: All usual residents aged 16 years and over',
   
]

# List of corresponding values 
value_cols = [

    # Housing / Accommodation
    'Accommodation type: Detached',
    'Accommodation type: Semi-detached',
    'Accommodation type: Terraced',
    'Accommodation type: In a purpose-built block of flats or tenement',

    # Deprivation
    'Household deprivation: Household is deprived in three dimensions; measures: Value',

    # Commuting
    'Distance travelled to work: Less than 2km',
    'Distance travelled to work: 10km to less than 20km',
    'Distance travelled to work: Works mainly at an offshore installation, in no fixed place, or outside the UK',

    # Tenure
    'Tenure of household: Social rented: Rents from council or Local Authority',
    'Tenure of household: Private rented',

    # Health
    'General health: Very bad health',

    # Socioeconomic Classification
    'National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers',
    'National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed',
    'National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students',
    
    # Language / Migration
    'Proficiency in English language: Main language is not English (English or Welsh in Wales): Cannot speak English well',
    
    # Age
    'Age: Aged 25 to 29 years',
    'Age: Aged 30 to 34 years',
    'Age: Aged 35 to 39 years', 
    'Age: Aged 60 to 64 years',

    # Economic Activity
    'Economic activity status: Economically inactive: Retired',
    'Economic activity status: Economically inactive: Long-term sick or disabled',

    # Ethnicity
    'Ethnic group: White',
    'Ethnic group: Black, Black British, Black Welsh, Caribbean or African',
    'Ethnic group: Asian, Asian British or Asian Welsh',

    # Transport
    'Number of cars or vans: No cars or vans in household',

    # Education
    'Highest level of qualification: No qualifications',

]

# Apply function to calculate percentages
result_dataframe = calculate_percentages(crime_SES_final, total_cols, value_cols)

In [ ]:
# Concatenate the new percentage columns with the original dataset
concatenated_df = pd.concat([crime_SES_final, result_dataframe], axis=1, ignore_index=False)

In [ ]:
# Subsetting the attributes we need - keep only the Output areas, geometry and percentages
keep_cols= [
 'OA21CD',
 'geometry',
 'Accommodation type: Detached_percentage',
 'Accommodation type: Semi-detached_percentage',
 'Accommodation type: Terraced_percentage',
 'Accommodation type: In a purpose-built block of flats or tenement_percentage',
 'Household deprivation: Household is deprived in three dimensions; measures: Value_percentage',
 'Distance travelled to work: Less than 2km_percentage',
 'Distance travelled to work: 10km to less than 20km_percentage',
 'Distance travelled to work: Works mainly at an offshore installation, in no fixed place, or outside the UK_percentage',
 'Tenure of household: Social rented: Rents from council or Local Authority_percentage',
 'Tenure of household: Private rented_percentage',
 'General health: Very bad health_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed_percentage',
 'National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students_percentage',
 'Proficiency in English language: Main language is not English (English or Welsh in Wales): Cannot speak English well_percentage',
 'Age: Aged 25 to 29 years_percentage',
 'Age: Aged 30 to 34 years_percentage',
 'Age: Aged 35 to 39 years_percentage',
 'Age: Aged 60 to 64 years_percentage',
 'Economic activity status: Economically inactive: Retired_percentage',
 'Economic activity status: Economically inactive: Long-term sick or disabled_percentage',
 'Ethnic group: White_percentage',
 'Ethnic group: Black, Black British, Black Welsh, Caribbean or African_percentage',
 'Ethnic group: Asian, Asian British or Asian Welsh_percentage',
 'Number of cars or vans: No cars or vans in household_percentage',
 'Highest level of qualification: No qualifications_percentage',
]

london_data = concatenated_df[keep_cols]

In [ ]:
# For more easy manipulation I define short column names

short_column_names = {
    'Accommodation type: Detached_percentage': 'Detached',
    'Accommodation type: Semi-detached_percentage': 'SemiDetached',
    'Accommodation type: Terraced_percentage': 'Terraced',
    'Accommodation type: In a purpose-built block of flats or tenement_percentage': 'Flats',
    'Household deprivation: Household is deprived in three dimensions; measures: Value_percentage': 'Deprivation3',
    'Distance travelled to work: Less than 2km_percentage': 'Commute_LT2km',
    'Distance travelled to work: 10km to less than 20km_percentage': 'Commute_10to20km',
    'Distance travelled to work: Works mainly at an offshore installation, in no fixed place, or outside the UK_percentage': 'Commute_Offshore',
    'Tenure of household: Social rented: Rents from council or Local Authority_percentage': 'SocialRent',
    'Tenure of household: Private rented_percentage': 'PrivateRent',
    'General health: Very bad health_percentage': 'BadHealth',
    'National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations_percentage': 'NSSEC_HighManager',
    'National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations_percentage': 'NSSEC_LowManager',
    'National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations_percentage': 'NSSEC_Intermediate',
    'National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers_percentage': 'NSSEC_SmallEmployers',
    'National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations_percentage': 'NSSEC_Tech',
    'National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations_percentage': 'NSSEC_SemiRoutine',
    'National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations_percentage': 'NSSEC_Routine',
    'National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed_percentage': 'NSSEC_Unemployed',
    'National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students_percentage': 'NSSEC_Students',
    'Proficiency in English language: Main language is not English (English or Welsh in Wales): Cannot speak English well_percentage': 'EngPoor',
    'Age: Aged 25 to 29 years_percentage': 'Age25_29',
    'Age: Aged 30 to 34 years_percentage': 'Age30_34',
    'Age: Aged 35 to 39 years_percentage': 'Age35_39',
    'Age: Aged 60 to 64 years_percentage': 'Age60_64',
    'Economic activity status: Economically inactive: Retired_percentage': 'Retired',
    'Economic activity status: Economically inactive: Long-term sick or disabled_percentage': 'SickDisabled',
    'Ethnic group: White_percentage': 'Ethnicity_White',
    'Ethnic group: Black, Black British, Black Welsh, Caribbean or African_percentage': 'Ethnicity_Black',
    'Ethnic group: Asian, Asian British or Asian Welsh_percentage': 'Ethnicity_Asian',
    'Number of cars or vans: No cars or vans in household_percentage': 'NoCars',
    'Highest level of qualification: No qualifications_percentage': 'NoQual',
}

london_data = london_data.rename(columns=short_column_names)

In [ ]:
# Select only numeric columns (percentage variables) for standardisation
# This excludes identifiers like 'OA_SA' or geometry which cannot be standardised
numeric_columns = london_data.select_dtypes(include='float64')

# Apply z-score standardisation to each variable
# This ensures variables are on the same scale (mean = 0, std = 1), which is essential before clustering
z_score_df = (numeric_columns - numeric_columns.mean()) / numeric_columns.std(ddof=0)

In [ ]:
# Compute the correlation matrix to assess multicollinearity between variables
corr = z_score_df.corr()

# Visualise correlations with a colour gradient to quickly identify strong relationships
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
# Define a correlation threshold (0.7) to flag highly correlated variable pairs.
# I want to identify variables with strong linear relationships (excluding perfect correlation = 1).
threshold = 0.7
highly_correlated = (corr.abs() > threshold) & (corr.abs() < 1.0)

# Plot a heatmap showing which variable pairs exceed the threshold.
# The result is a binary matrix (True/False), showing multicollinearity visually.
plt.figure(figsize=(10, 15))
sns.heatmap(highly_correlated, cmap='coolwarm', cbar=False, annot=True)
plt.title('Highly Correlated Variables')
plt.show()

In [ ]:
# Based on the heatmap and correlation matrix, drop selected variables to reduce redundancy
# Variables were removed due to strong correlations with others or for practical redundancy in clustering
z_score_df.drop(['NoQual',
                 'Commute_Offshore', 
                 'NSSEC_Routine',
                 'NSSEC_SemiRoutine'], axis=1, inplace=True)

In [ ]:
# Recalculate the correlation matrix and re-plot to check if multicollinearity has improved
corr_2 = z_score_df.corr()
corr_2.style.background_gradient(cmap='coolwarm')

In [ ]:
# Handle any remaining missing values by filling them with the variable's mean
z_score_df.fillna(z_score_df.mean(), inplace=True)

In [ ]:
# Use the elbow method to plot within-cluster sum of squares (inertia) for different k values
Sum_of_squared_distances = []

K_range = range(1,15)

for k in K_range:
 km = KMeans(n_clusters=k)
 km = km.fit(z_score_df)
 Sum_of_squared_distances.append(km.inertia_)
    
plt.plot(K_range, Sum_of_squared_distances, 'bx-')
plt.xlabel('k')
plt.ylabel('Sum_of_squared_distances')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
# Plot between-cluster sum of squares to understand how discriminatory the models are when different numbers of groups (k) 
# are produced
def elbow(dataframe, n):
    kMeansVar = [KMeans(n_clusters=k).fit(dataframe.values) for k in range(1, n)] #making use of list comprehension
    centroids = [X.cluster_centers_ for X in kMeansVar]
    k_euclid = [cdist(dataframe.values, cent) for cent in centroids]
    dist = [np.min(ke, axis=1) for ke in k_euclid]
    wcss = [sum(d**2) for d in dist]
    tss = sum(pdist(dataframe.values)**2)/dataframe.values.shape[0]
    bss = tss - wcss
    plt.plot(bss)
    plt.show()
 
elbow(z_score_df,15)

In [ ]:
# Fit final model using the selected number of clusters (k = 4)
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(z_score_df)
labels = kmeans.predict(z_score_df)
cluster_centres = kmeans.cluster_centers_

z_score_df['Cluster'] = kmeans.labels_

In [ ]:
# Alternative visualisation using Seaborn - static figure with the point variability included in the x/y-axis label
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(z_score_df)

z_score_df['Cluster'] = clusters

# Standardize the data for PCA
scaler = StandardScaler()
stand_data_scaled = scaler.fit_transform(z_score_df)

# PCA
pca = PCA(n_components=2).fit(stand_data_scaled)
pca_result = pca.transform(stand_data_scaled)

#Percentage of variance explained by each of the selected components
variance_ratio = pca.explained_variance_ratio_

plt.figure(figsize=(10, 6))
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=clusters, palette='viridis', s=50, alpha=0.7)
plt.title('Cluster Plot against 1st 2 Principal Components')
plt.xlabel(f'Principal Component 1 variation: {variance_ratio[0]*100:.2f}%')
plt.ylabel(f'Principal Component 2 variation: {variance_ratio[1]*100:.2f}%')
plt.legend(title='Clusters')
plt.show()


In [ ]:
# Perform KMeans clustering with 4 groups
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(z_score_df)

# Extract cluster centres as a DataFrame (still in Z-score units)
cluster_centers = kmeans.cluster_centers_

cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=z_score_df.columns)

# Display the first few cluster centres
cluster_centers.head()

In [ ]:
# Select the centre values for Cluster 0 (the first cluster)
first_row_centers = cluster_centers.iloc[0, :]

# Count the number of features (i.e., variables used in clustering)
num_features = len(first_row_centers)

# Generate evenly spaced angles (in radians) for each feature around a circle
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)

# Create a polar plot using matplotlib
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))

# Plot the cluster centre values as a line on the radar chart
ax.plot(theta, first_row_centers, linewidth=2, color='blue', marker='o', label='Centres')

# Plot a red baseline representing the origin (zero line)
ax.plot(theta, np.zeros_like(first_row_centers), color='red', linestyle='--', label='Average')

# Label each spoke of the radar chart with the corresponding variable name
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=6)

# Add a legend and tidy layout
plt.title("Cluster 0 Profile Across Census Variables", fontsize=12, pad=20)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 1 
second_row_centers = cluster_centers.iloc[1, :] 
num_features = len(second_row_centers)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, second_row_centers, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(second_row_centers), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 1 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 2
third_row_centers = cluster_centers.iloc[2, :] 
num_features = len(third_row_centers)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, third_row_centers, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(third_row_centers), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 2 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 3
fourth_row_centers = cluster_centers.iloc[3, :] 
num_features = len(fourth_row_centers)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, fourth_row_centers, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(fourth_row_centers), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 3 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Drop the original census variables used for clustering
# These are no longer needed now that clustering is complete
z_score_df.drop([
 'Detached',
 'SemiDetached',
 'Terraced',
 'Flats',
 'Commute_LT2km',
 'Commute_10to20km',
 'SocialRent',
 'PrivateRent',
 'NSSEC_HighManager',
 'NSSEC_LowManager',
 'NSSEC_Intermediate',
 'NSSEC_SmallEmployers',
 'NSSEC_Tech',
 'NSSEC_Unemployed',
 'NSSEC_Students',
 'Age25_29',
 'Age30_34',
 'Age35_39',
 'Age60_64',
 'Retired',
 'SickDisabled',
 'Ethnicity_White',
 'Ethnicity_Black',
 'Ethnicity_Asian',
 'NoCars'], axis=1, inplace=True)

In [ ]:
# Concatenate the resulting tables - combine the original census dataset (which includes spatial geometry)
# with the cluster labels stored in z_score_df
# This results in a single DataFrame that contains both spatial and cluster information
final_df = pd.concat([london_data, z_score_df], axis=1, ignore_index=False)

In [ ]:
# Check column names to ensure everything has merged correctly
final_df.columns

In [ ]:
# Create an interactive map to visualise the clusters across London
# Each cluster is shown in a different colour
final_df.explore(column='Cluster', cmap='Set1', tiles='CartoDB positron')

In [ ]:
# Define a function to rename cluster numbers with updated, concise labels
def rename_column(x): 
    x = str(x)  # Convert integer to string
    x = x.replace("0", "0")
    x = x.replace("1", "1")
    x = x.replace("2", "2")
    x = x.replace("3", "3")
    return x

# Apply the renaming function to the 'Cluster' column
final_df['Cluster'] = final_df['Cluster'].apply(rename_column)

In [ ]:
# Create a final interactive map using the renamed cluster labels
final_df.explore(column='Cluster', cmap='Set1', tiles='CartoDB positron')

#### Geodemographic Classification (Crime)

In [ ]:
# Function to calculate percentage values for selected demographic and socio-economic indicators
# Each value column is divided by its corresponding total column to compute a new percentage feature
def calculate_percentages(dataframe, total_columns, value_columns):

    result_df = pd.DataFrame()

    for total_col, value_col in zip(total_columns, value_columns):
        percentage_col_name = f"{value_col}_percentage"

        if total_col not in dataframe.columns or value_col not in dataframe.columns:
            print(f"Warning: '{total_col}' or '{value_col}' not found in DataFrame. Skipping...")
            continue  # Skips missing columns instead of raising an error

        # Convert columns to numeric or NaN if errors occur
        dataframe[value_col] = pd.to_numeric(dataframe[value_col], errors='coerce')
        dataframe[total_col] = pd.to_numeric(dataframe[total_col], errors='coerce')
        
        result_df[percentage_col_name] = (dataframe[value_col] / dataframe[total_col]) * 100

    return result_df

# List of the corresponding totals 
total_cols = [
    'Total_Crime',
    'Total_Crime',
    'Total_Crime',
    'Total_Crime'
]

# List of corresponding values 
value_cols = [
    'Anti-social Behaviours',
    'Drug-related Crime',
    'Property-Related Crime',
    'Violence against the person'
]

# Apply function to calculate percentages
result_dataframe_crime = calculate_percentages(crime_SES_final, total_cols, value_cols)

In [ ]:
# Concatenate the new percentage columns with the original dataset
concatenated_df_crime = pd.concat([crime_SES_final, result_dataframe_crime], axis=1, ignore_index=False)

In [ ]:
# Subsetting the attributes we need - keep only the Output areas, geometry and percentages
keep_cols= [
 'OA21CD',
 'geometry',
 'Anti-social Behaviours_percentage',
 'Drug-related Crime_percentage',
 'Property-Related Crime_percentage',
 'Violence against the person_percentage'
]

london_data_crime = concatenated_df_crime[keep_cols]

In [ ]:
# For more easy manipulation I define short column names
short_column_names = {
    'Anti-social Behaviours_percentage': 'AntisocialCrime',
    'Drug-related Crime_percentage': 'DrugCrime',
    'Property-Related Crime_percentage': 'PropertyCrime',
    'Violence against the person_percentage': 'ViolentCrime'
}

london_data_crime = london_data_crime.rename(columns=short_column_names)

In [ ]:
# Select only numeric columns (percentage variables) for standardisation
# This excludes identifiers like 'OA_SA' or geometry which cannot be standardised
numeric_columns = london_data_crime.select_dtypes(include='float64')

# Apply z-score standardisation to each variable
# This ensures variables are on the same scale (mean = 0, std = 1), which is essential before clustering
z_score_df_crime = (numeric_columns - numeric_columns.mean()) / numeric_columns.std(ddof=0)

In [ ]:
# Compute the correlation matrix to assess multicollinearity between variables
corr = z_score_df_crime.corr()

# Visualise correlations with a colour gradient to quickly identify strong relationships
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
# Handle any remaining missing values by filling them with the variable's mean
z_score_df_crime.fillna(z_score_df_crime.mean(), inplace=True)

In [ ]:
# Use the elbow method to plot within-cluster sum of squares (inertia) for different k values
Sum_of_squared_distances = []

K_range = range(1,15)

for k in K_range:
 km = KMeans(n_clusters=k)
 km = km.fit(z_score_df_crime)
 Sum_of_squared_distances.append(km.inertia_)
    
plt.plot(K_range, Sum_of_squared_distances, 'bx-')
plt.xlabel('k')
plt.ylabel('Sum_of_squared_distances')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
# Plot between-cluster sum of squares to understand how discriminatory the models are when different numbers of groups (k) 
# are produced
def elbow(dataframe, n):
    kMeansVar = [KMeans(n_clusters=k).fit(dataframe.values) for k in range(1, n)] #making use of list comprehension
    centroids = [X.cluster_centers_ for X in kMeansVar]
    k_euclid = [cdist(dataframe.values, cent) for cent in centroids]
    dist = [np.min(ke, axis=1) for ke in k_euclid]
    wcss = [sum(d**2) for d in dist]
    tss = sum(pdist(dataframe.values)**2)/dataframe.values.shape[0]
    bss = tss - wcss
    plt.plot(bss)
    plt.show()
 
elbow(z_score_df_crime,15)

In [ ]:
# Alternative visualisation using Seaborn - static figure with the point variability included in the x/y-axis label
kmeans = KMeans(n_clusters=5, random_state=50)
clusters = kmeans.fit_predict(z_score_df_crime)

z_score_df_crime['Cluster'] = clusters

# Standardize the data for PCA
scaler = StandardScaler()
stand_data_scaled = scaler.fit_transform(z_score_df_crime)

# PCA
pca = PCA(n_components=2).fit(stand_data_scaled)
pca_result = pca.transform(stand_data_scaled)

#Percentage of variance explained by each of the selected components
variance_ratio = pca.explained_variance_ratio_

plt.figure(figsize=(10, 6))
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=clusters, palette='viridis', s=50, alpha=0.7)
plt.title('Cluster Plot against 1st 2 Principal Components')
plt.xlabel(f'Principal Component 1 variation: {variance_ratio[0]*100:.2f}%')
plt.ylabel(f'Principal Component 2 variation: {variance_ratio[1]*100:.2f}%')
plt.legend(title='Clusters')
plt.show()

In [ ]:
# Perform KMeans clustering with 4 groups
kmeans = KMeans(n_clusters=5, random_state=50)
clusters = kmeans.fit_predict(z_score_df_crime)

# Extract cluster centres as a DataFrame (still in Z-score units)
cluster_centers = kmeans.cluster_centers_

cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=z_score_df_crime.columns)

# Display the first few cluster centres
cluster_centers.head()

In [ ]:
# Select the centre values for Cluster 0 (the first cluster)
first_row_centers_crimes = cluster_centers.iloc[0, :]

# Count the number of features (i.e., variables used in clustering)
num_features = len(first_row_centers_crimes)

# Generate evenly spaced angles (in radians) for each feature around a circle
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)

# Create a polar plot using matplotlib
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))

# Plot the cluster centre values as a line on the radar chart
ax.plot(theta, first_row_centers_crimes, linewidth=2, color='blue', marker='o', label='Centres')

# Plot a red baseline representing the origin (zero line)
ax.plot(theta, np.zeros_like(first_row_centers_crimes), color='red', linestyle='--', label='Average')

# Label each spoke of the radar chart with the corresponding variable name
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=6)

# Add a legend and tidy layout
plt.title("Cluster 0 Profile Across Census Variables", fontsize=12, pad=20)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 1 
second_row_centers_crime = cluster_centers.iloc[1, :] 
num_features = len(second_row_centers_crime)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, second_row_centers_crime, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(second_row_centers_crime), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 1 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 2
third_row_centers_crime = cluster_centers.iloc[2, :] 
num_features = len(third_row_centers_crime)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, third_row_centers_crime, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(third_row_centers_crime), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 2 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 3
fourth_row_centers_crime = cluster_centers.iloc[3, :] 
num_features = len(fourth_row_centers_crime)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, fourth_row_centers_crime, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(fourth_row_centers_crime), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 3 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Follow the same structure as Cluster 0 but for Cluster 3
fifth_row_centers_crime = cluster_centers.iloc[4, :] 
num_features = len(fifth_row_centers_crime)
theta = np.linspace(0, 2 * np.pi, num_features, endpoint=True)
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 6))
ax.plot(theta, fifth_row_centers_crime, linewidth=2, color='blue', marker='o', label='Centers')
ax.plot(theta, np.zeros_like(fifth_row_centers_crime), color='red', linestyle='--', label='Average')
ax.set_xticks(theta)
ax.set_xticklabels(cluster_centers.columns, rotation=45, ha='right', fontsize=8)
plt.title("Cluster 4 Profile Across Census Variables", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Drop the original census variables used for clustering
# These are no longer needed now that clustering is complete
z_score_df_crime.drop([
 'AntisocialCrime',
 'DrugCrime',
 'PropertyCrime',
 'ViolentCrime'], axis=1, inplace=True)

In [ ]:
# Concatenate the resulting tables - combine the original census dataset (which includes spatial geometry)
# with the cluster labels stored in z_score_df
# This results in a single DataFrame that contains both spatial and cluster information
final_df_crime = pd.concat([london_data_crime, z_score_df_crime], axis=1, ignore_index=False)

In [ ]:
# Create an interactive map to visualise the clusters across London
# Each cluster is shown in a different colour
final_df_crime.explore(column='Cluster', cmap='Set1', tiles='CartoDB positron')

In [ ]:
# Define a function to rename cluster numbers with updated, concise labels
def rename_column(x): 
    x = str(x)  # Convert integer to string
    x = x.replace("0", "High Property, Low Drug and Antisocial Crime Area")
    x = x.replace("1", "High Drug, Low Property Crime Area")
    x = x.replace("2", "High Antisocial, Low Property Crime Area")
    x = x.replace("3", "High Property, Low Violent Crime Area")
    x = x.replace("4", "High Violent, Low Property Crime Area")
    return x

# Apply the renaming function to the 'Cluster' column
final_df_crime['Cluster'] = final_df_crime['Cluster'].apply(rename_column)

In [ ]:
# Create a final interactive map using the renamed cluster labels
final_df_crime.explore(column='Cluster', cmap='Set1', tiles='CartoDB positron')

### Geodemographic Results: 
- Visual comparison of cluster types and crime concentrations.
- Description of crime type distributions across clusters.
- Boxplots showing differences in crime rates between clusters.

In [ ]:
value_cols = ['Accommodation type: Detached',
    'Accommodation type: Semi-detached',
    'Accommodation type: Terraced',
    'Accommodation type: In a purpose-built block of flats or tenement',
    'Distance travelled to work: Less than 2km',
    'Distance travelled to work: 10km to less than 20km',
    'Distance travelled to work: Works mainly at an offshore installation, in no fixed place, or outside the UK',
    'Tenure of household: Social rented: Rents from council or Local Authority',
    'Tenure of household: Private rented',
    'National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers',
    'National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations',
    'National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed',
    'National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students',
    'Proficiency in English language: Main language is not English (English or Welsh in Wales): Cannot speak English well',
    'Age: Aged 25 to 29 years',
    'Age: Aged 30 to 34 years',
    'Age: Aged 35 to 39 years', 
    'Age: Aged 60 to 64 years',
    'Economic activity status: Economically inactive: Retired',
    'Economic activity status: Economically inactive: Long-term sick or disabled',
    'Ethnic group: White',
    'Ethnic group: Black, Black British, Black Welsh, Caribbean or African',
    'Ethnic group: Asian, Asian British or Asian Welsh',
    'Number of cars or vans: No cars or vans in household',
    'Highest level of qualification: No qualifications',
    'Anti-social Behaviours',
    'Drug-related Crime',
    'Property-Related Crime',
    'Violence against the person'
]

In [ ]:
# Summary statistics for selected variables 
crime_SES_final[value_cols].describe()

In [ ]:
violin_vars = [
    'Accommodation type: Detached',
    'Accommodation type: In a purpose-built block of flats or tenement',
    'Economic activity status: Economically inactive: Retired',
    'National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations',
    'Ethnic group: Black, Black British, Black Welsh, Caribbean or African',
    'Proficiency in English language: Main language is not English (English or Welsh in Wales): Cannot speak English well'
]

# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2 rows, 3 columns
fig.suptitle('Distribution of Selected Variables (Violin Plots)', fontsize=18, weight='bold')

for ax, var in zip(axes.flatten(), violin_vars):
    sns.violinplot(data=final_data, y=var, ax=ax, color="lightblue")
    ax.set_title(var, fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('Percentage')

plt.tight_layout(rect=[0, 0, 1, 0.95])  # Make space for the main title
plt.show()


In [ ]:
hist_vars = [
    'Distance travelled to work: Less than 2km',
    'Distance travelled to work: Works mainly at an offshore installation, in no fixed place, or outside the UK',
    'Tenure of household: Private rented',
    'Age: Aged 25 to 29 years',
    'Ethnic group: Asian, Asian British or Asian Welsh',
    'Highest level of qualification: No qualifications'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Selected Variables (Histograms)', fontsize=18, weight='bold')

for ax, var in zip(axes.flatten(), hist_vars):
    sns.histplot(data=final_data, x=var, bins=30, kde=True, ax=ax, color="salmon")
    ax.set_title(var, fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('Count')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# Assuming 'LSOA11CD' is the key in both
gdf = final_df.merge(crime_groups_OA, on="OA21CD")

In [ ]:
# Group by cluster and calculate average crime
crime_by_cluster = gdf.groupby("Cluster")["Total_Crime"].mean().reset_index()
print(crime_by_cluster)

In [ ]:
# Group by 'Cluster' and apply descriptive statistics
crime_stats = gdf.groupby("Cluster")["Total_Crime"].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max'
).reset_index()

print(crime_stats)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(crime_by_cluster["Cluster"], crime_by_cluster["Total_Crime"])
plt.xlabel("Cluster")
plt.ylabel("Average Crime Count")
plt.title("Average Crime by Geodemographic Cluster")
plt.show()

In [ ]:
crime_groups_OA.head()

In [ ]:
crime_type_cols = [ 'Anti-social Behaviours', 'Drug-related Crime', 'Property-Related Crime', 'Violence against the person']

# Mean crime per cluster
crime_means = gdf.groupby("Cluster")[crime_type_cols].mean().round(2)
crime_means


In [ ]:
import matplotlib.pyplot as plt

# Transpose for a cleaner bar plot (crime types on x-axis)
crime_means.T.plot(kind="bar", figsize=(14, 6))
plt.title("Average Number of Crimes by Type per Cluster")
plt.ylabel("Mean Number of Crimes")
plt.xlabel("Crime Type")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Step 1: Define the high-crime areas based on 75th percentile (top 25%)
total_crime_threshold = gdf["Total_Crime"].quantile(0.75)
violence_threshold = gdf["Violence against the person"].quantile(0.75)
ASB_threshold = gdf["Anti-social Behaviours"].quantile(0.75)
property_threshold = gdf["Property-Related Crime"].quantile(0.75)
drug_threshold = gdf["Drug-related Crime"].quantile(0.75)

# Step 2: Filter areas
high_total_crime = gdf[gdf["Total_Crime"] > total_crime_threshold]
high_violence = gdf[gdf["Violence against the person"] > violence_threshold]
high_ASB = gdf[gdf["Anti-social Behaviours"] > ASB_threshold]
high_property = gdf[gdf["Property-Related Crime"] > property_threshold]
high_drugs = gdf[gdf["Drug-related Crime"] > drug_threshold]

# Step 1: Define the low-crime thresholds (bottom 25%)
total_crime_threshold = gdf["Total_Crime"].quantile(0.25)
violence_threshold = gdf["Violence against the person"].quantile(0.25)
ASB_threshold = gdf["Anti-social Behaviours"].quantile(0.25)
property_threshold = gdf["Property-Related Crime"].quantile(0.25)
drug_threshold = gdf["Drug-related Crime"].quantile(0.25)

# Step 2: Filter low-crime areas
low_total_crime = gdf[gdf["Total_Crime"] < total_crime_threshold]
low_violence = gdf[gdf["Violence against the person"] < violence_threshold]
low_ASB = gdf[gdf["Anti-social Behaviours"] < ASB_threshold]
low_property = gdf[gdf["Property-Related Crime"] < property_threshold]
low_drugs = gdf[gdf["Drug-related Crime"] < drug_threshold]


In [ ]:
import matplotlib.pyplot as plt

# Set up side-by-side subplots
fig, axs = plt.subplots(1, 2, figsize=(24, 12))  # 1 row, 2 columns
axs = axs.flatten()

# Plot 1: Low Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[0], 
    legend=True, 
    edgecolor='white', 
    linewidth=0.2
)
low_total_crime.plot(
    ax=axs[0], 
    facecolor='none', 
    edgecolor='blue',  # different colour for low crime
    linewidth=0.7, 
    alpha=0.6
)
axs[0].set_title('SES Clusters with Low Total Crime Overlays', fontsize=16)
axs[0].axis('off')

# Plot 2: High Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[1], 
    legend=False,  # Legend only once
    edgecolor='white', 
    linewidth=0.2
)
high_total_crime.plot(
    ax=axs[1], 
    facecolor='none', 
    edgecolor='red', 
    linewidth=0.7, 
    alpha=0.6
)
axs[1].set_title('SES Clusters with High Total Crime Overlays', fontsize=16)
axs[1].axis('off')

# Tight layout for spacing
plt.tight_layout()
plt.show()


In [ ]:
# Set up side-by-side subplots
fig, axs = plt.subplots(1, 2, figsize=(24, 12))  # 1 row, 2 columns
axs = axs.flatten()

# Plot 1: Low Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[0], 
    legend=True, 
    edgecolor='white', 
    linewidth=0.2
)
low_violence.plot(
    ax=axs[0], 
    facecolor='none', 
    edgecolor='blue',  # different colour for low crime
    linewidth=0.7, 
    alpha=0.6
)
axs[0].set_title('SES Clusters with Low Violence Crime Overlays', fontsize=16)
axs[0].axis('off')

# Plot 2: High Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[1], 
    legend=False,  # Legend only once
    edgecolor='white', 
    linewidth=0.2
)
high_violence.plot(
    ax=axs[1], 
    facecolor='none', 
    edgecolor='red', 
    linewidth=0.7, 
    alpha=0.6
)
axs[1].set_title('SES Clusters with High Violence Crime Overlays', fontsize=16)
axs[1].axis('off')

# Tight layout for spacing
plt.tight_layout()
plt.show()


In [ ]:
# Set up side-by-side subplots
fig, axs = plt.subplots(1, 2, figsize=(24, 12))  # 1 row, 2 columns
axs = axs.flatten()

# Plot 1: Low Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[0], 
    legend=True, 
    edgecolor='white', 
    linewidth=0.2
)
low_ASB.plot(
    ax=axs[0], 
    facecolor='none', 
    edgecolor='blue',  # different colour for low crime
    linewidth=0.7, 
    alpha=0.6
)
axs[0].set_title('SES Clusters with Low ASB Crime Overlays', fontsize=16)
axs[0].axis('off')

# Plot 2: High Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[1], 
    legend=False,  # Legend only once
    edgecolor='white', 
    linewidth=0.2
)
high_ASB.plot(
    ax=axs[1], 
    facecolor='none', 
    edgecolor='red', 
    linewidth=0.7, 
    alpha=0.6
)
axs[1].set_title('SES Clusters with High ASB Crime Overlays', fontsize=16)
axs[1].axis('off')

# Tight layout for spacing
plt.tight_layout()
plt.show()


In [ ]:
# Set up side-by-side subplots
fig, axs = plt.subplots(1, 2, figsize=(24, 12))  # 1 row, 2 columns
axs = axs.flatten()

# Plot 1: Low Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[0], 
    legend=True, 
    edgecolor='white', 
    linewidth=0.2
)
low_property.plot(
    ax=axs[0], 
    facecolor='none', 
    edgecolor='blue',  # different colour for low crime
    linewidth=0.7, 
    alpha=0.6
)
axs[0].set_title('SES Clusters with Low Property Crime Overlays', fontsize=16)
axs[0].axis('off')

# Plot 2: High Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[1], 
    legend=False,  # Legend only once
    edgecolor='white', 
    linewidth=0.2
)
high_property.plot(
    ax=axs[1], 
    facecolor='none', 
    edgecolor='red', 
    linewidth=0.7, 
    alpha=0.6
)
axs[1].set_title('SES Clusters with High Property Crime Overlays', fontsize=16)
axs[1].axis('off')

# Tight layout for spacing
plt.tight_layout()
plt.show()


In [ ]:
# Set up side-by-side subplots
fig, axs = plt.subplots(1, 2, figsize=(24, 12))  # 1 row, 2 columns
axs = axs.flatten()

# Plot 1: Low Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[0], 
    legend=True, 
    edgecolor='white', 
    linewidth=0.2
)
low_drugs.plot(
    ax=axs[0], 
    facecolor='none', 
    edgecolor='blue',  # different colour for low crime
    linewidth=0.7, 
    alpha=0.6
)
axs[0].set_title('SES Clusters with Low Drug Crime Overlays', fontsize=16)
axs[0].axis('off')

# Plot 2: High Total Crime
gdf.plot(
    column='Cluster', 
    cmap='Set3', 
    ax=axs[1], 
    legend=False,  # Legend only once
    edgecolor='white', 
    linewidth=0.2
)
high_drugs.plot(
    ax=axs[1], 
    facecolor='none', 
    edgecolor='red', 
    linewidth=0.7, 
    alpha=0.6
)
axs[1].set_title('SES Clusters with High Drug Crime Overlays', fontsize=16)
axs[1].axis('off')

# Tight layout for spacing
plt.tight_layout()
plt.show()
